In [2]:
!pip -q install gradio

In [3]:
import json
from abc import ABC, abstractmethod
class SimpleDate:
    @staticmethod
    def today_str() -> str:
        return "2026-09-28"

class Person(ABC):
    def __init__(self, person_id: int, name: str):
        if not name or not name.strip():
            raise ValueError("Name cannot be empty.")
        self.__person_id = person_id
        self.__name = name.strip()

    @property
    def person_id(self) -> int:
        return self.__person_id

    @property
    def name(self) -> str:
        return self.__name

    @abstractmethod
    def role(self) -> str:
        pass


class User(Person):
    MAX_BORROWING = 3

    def role(self) -> str:
        return "User"


class Author(Person):
    def role(self) -> str:
        return "Author"


class Category:
    def __init__(self, category_id: int, name: str):
        if not name or not name.strip():
            raise ValueError("Category name cannot be empty.")
        self.category_id = category_id
        self.name = name.strip()


class Book:
    def __init__(self, book_id: int, title: str, author_id: int, category_id: int, copies: int):
        if copies <= 0:
            raise ValueError("Copies must be greater than zero.")
        if not title or not title.strip():
            raise ValueError("Title cannot be empty.")

        self.__book_id = book_id
        self.__title = title.strip()
        self.__author_id = author_id
        self.__category_id = category_id
        self.__copies = copies
        self.__available_copies = copies

    @property
    def book_id(self) -> int:
        return self.__book_id

    @property
    def title(self) -> str:
        return self.__title

    @title.setter
    def title(self, value: str):
        if not value or not value.strip():
            raise ValueError("Title cannot be empty.")
        self.__title = value.strip()

    @property
    def author_id(self) -> int:
        return self.__author_id

    @property
    def category_id(self) -> int:
        return self.__category_id

    @property
    def copies(self) -> int:
        return self.__copies

    @property
    def available_copies(self) -> int:
        return self.__available_copies

    @available_copies.setter
    def available_copies(self, value: int):
        if value < 0 or value > self.__copies:
            raise ValueError("Invalid available copies value.")
        self.__available_copies = value

    def borrow(self):
        if self.__available_copies == 0:
            raise ValueError("No copies are available.")
        self.__available_copies -= 1

    def return_copy(self):
        if self.__available_copies == self.__copies:
            raise ValueError("All copies are already available.")
        self.__available_copies += 1


class Borrowing:
    def __init__(self, borrowing_id: int, user_id: int, book_id: int, borrow_date: str, return_date: str = None):
        self.borrowing_id = borrowing_id
        self.user_id = user_id
        self.book_id = book_id
        self.borrow_date = borrow_date
        self.return_date = return_date

    @property
    def active(self) -> bool:
        return self.return_date is None

    def return_book(self):
        self.return_date = SimpleDate.today_str()


class FileManager:
    FILE_NAME = "library_data.json"

    @staticmethod
    def save(data: dict):
        with open(FileManager.FILE_NAME, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=4)

    @staticmethod
    def load() -> dict:
        try:
            with open(FileManager.FILE_NAME, "r", encoding="utf-8") as file:
                return json.load(file)
        except (FileNotFoundError, json.JSONDecodeError):
            return {
                "users": [],
                "authors": [],
                "categories": [],
                "books": [],
                "borrowings": []
            }

class Library:
    def __init__(self):
        self.users: list[User] = []
        self.authors: list[Author] = []
        self.categories: list[Category] = []
        self.books: list[Book] = []
        self.borrowings: list[Borrowing] = []

        self.load_data()

    def next_id(self, objects, attribute):
        if not objects:
            return 1
        return max(getattr(obj, attribute) for obj in objects) + 1

    def add_user(self, name: str) -> User:
        user = User(self.next_id(self.users, "person_id"), name)
        self.users.append(user)
        self.save_data()
        return user

    def get_user(self, user_id: int) -> User:
        for user in self.users:
            if user.person_id == user_id:
                return user
        raise ValueError("User not found.")

    def add_author(self, name: str) -> Author:
        author = Author(self.next_id(self.authors, "person_id"), name)
        self.authors.append(author)
        self.save_data()
        return author

    def get_author(self, author_id: int) -> Author:
        for author in self.authors:
            if author.person_id == author_id:
                return author
        raise ValueError("Author not found.")

    def add_category(self, name: str) -> Category:
        category = Category(self.next_id(self.categories, "category_id"), name)
        self.categories.append(category)
        self.save_data()
        return category

    def get_category(self, category_id: int) -> Category:
        for category in self.categories:
            if category.category_id == category_id:
                return category
        raise ValueError("Category not found.")

    def add_book(self, title: str, author_id: int, category_id: int, copies: int) -> Book:
        self.get_author(author_id)
        self.get_category(category_id)

        book = Book(self.next_id(self.books, "book_id"), title, author_id, category_id, copies)
        self.books.append(book)
        self.save_data()
        return book

    def update_book(self, book_id: int, new_title: str):
        book = self.get_book(book_id)
        book.title = new_title
        self.save_data()

    def delete_book(self, book_id: int):
        book = self.get_book(book_id)
        if any(b.book_id == book_id and b.active for b in self.borrowings):
            raise ValueError("Cannot delete a borrowed book.")
        self.books.remove(book)
        self.save_data()

    def get_book(self, book_id: int) -> Book:
        for book in self.books:
            if book.book_id == book_id:
                return book
        raise ValueError("Book not found.")

    def search_books(self, keyword: str) -> list[Book]:
        keyword = keyword.lower().strip()
        return [
            book for book in self.books
            if keyword in book.title.lower() or keyword in self.get_author(book.author_id).name.lower()
        ]

    def borrow_book(self, user_id: int, book_id: int) -> Borrowing:
        user = self.get_user(user_id)
        book = self.get_book(book_id)

        active = [b for b in self.borrowings if b.user_id == user_id and b.active]

        if book.available_copies == 0:
            raise ValueError("No copies are available.")

        if any(b.book_id == book_id for b in active):
            raise ValueError("User cannot borrow the same book twice.")

        if len(active) >= User.MAX_BORROWING:
            raise ValueError("User reached the maximum borrowing limit.")

        book.borrow()
        borrowing = Borrowing(
            self.next_id(self.borrowings, "borrowing_id"),
            user.person_id,
            book.book_id,
            SimpleDate.today_str()
        )
        self.borrowings.append(borrowing)
        self.save_data()
        return borrowing

    def return_book(self, borrowing_id: int):
        borrowing = next((b for b in self.borrowings if b.borrowing_id == borrowing_id), None)
        if borrowing is None:
            raise ValueError("Borrowing not found.")
        if not borrowing.active:
            raise ValueError("Book is already returned.")

        book = self.get_book(borrowing.book_id)
        book.return_copy()
        borrowing.return_book()
        self.save_data()

    def report(self) -> dict:
        return {
            "Total Books": len(self.books),
            "Available Copies": sum(b.available_copies for b in self.books),
            "Users": len(self.users),
            "Active Borrowings": sum(1 for b in self.borrowings if b.active)
        }

    def save_data(self):
        data = {
            "users": [{"id": u.person_id, "name": u.name} for u in self.users],
            "authors": [{"id": a.person_id, "name": a.name} for a in self.authors],
            "categories": [{"id": c.category_id, "name": c.name} for c in self.categories],
            "books": [
                {
                    "id": b.book_id,
                    "title": b.title,
                    "author_id": b.author_id,
                    "category_id": b.category_id,
                    "copies": b.copies,
                    "available": b.available_copies
                }
                for b in self.books
            ],
            "borrowings": [
                {
                    "id": b.borrowing_id,
                    "user_id": b.user_id,
                    "book_id": b.book_id,
                    "borrow_date": b.borrow_date,
                    "return_date": b.return_date
                }
                for b in self.borrowings
            ]
        }
        FileManager.save(data)

    def load_data(self):
        data = FileManager.load()
        self.users = [User(x["id"], x["name"]) for x in data.get("users", [])]
        self.authors = [Author(x["id"], x["name"]) for x in data.get("authors", [])]
        self.categories = [Category(x["id"], x["name"]) for x in data.get("categories", [])]

        self.books = []
        for x in data.get("books", []):
            book = Book(x["id"], x["title"], x["author_id"], x["category_id"], x["copies"])
            book.available_copies = x["available"]
            self.books.append(book)

        self.borrowings = [
            Borrowing(x["id"], x["user_id"], x["book_id"], x["borrow_date"], x["return_date"])
            for x in data.get("borrowings", [])
        ]


def run_test_scenarios():
    print("\n========== STARTING AUTOMATED TEST SUITE ==========")
    test_lib = Library()
    test_lib.users, test_lib.authors, test_lib.categories, test_lib.books, test_lib.borrowings = [], [], [], [], []

    author = test_lib.add_author("Clean Code Author")
    category = test_lib.add_category("Computer Science")

    try:
        book = test_lib.add_book("Python Basics", author.person_id, category.category_id, 2)
        print("Test 1 (Add Book): PASS")
    except Exception as e:
        print(f"Test 1 (Add Book): FAIL - {e}")

    try:
        user = test_lib.add_user("Ahmed Mohamed")
        print("Test 2 (Add User): PASS")
    except Exception as e:
        print(f"Test 2 (Add User): FAIL - {e}")

    try:
        br = test_lib.borrow_book(user.person_id, book.book_id)
        print("Test 3 (Borrow Book): PASS")
    except Exception as e:
        print(f"Test 3 (Borrow Book): FAIL - {e}")

    try:
        test_lib.add_book("", author.person_id, category.category_id, 5)
        print("Test 4 (Empty Title): FAIL")
    except ValueError:
        print("Test 4 (Empty Title Validation): PASS")

    try:
        test_lib.add_book("Invalid Copies Book", author.person_id, category.category_id, 0)
        print("Test 5 (Invalid Copies): FAIL")
    except ValueError:
        print("Test 5 (Invalid Copies Validation): PASS")

    try:
        test_lib.borrow_book(999, book.book_id)
        print("Test 6 (Invalid User ID): FAIL")
    except ValueError:
        print("Test 6 (Invalid User Validation): PASS")

    try:
        test_lib.borrow_book(user.person_id, book.book_id)
        print("Test 7 (Borrow Same Book Twice): FAIL")
    except ValueError:
        print("Test 7 (Borrow Same Book Twice Rule): PASS")

    try:
        b2 = test_lib.add_book("Book 2", author.person_id, category.category_id, 2)
        b3 = test_lib.add_book("Book 3", author.person_id, category.category_id, 2)
        b4 = test_lib.add_book("Book 4", author.person_id, category.category_id, 2)

        test_lib.borrow_book(user.person_id, b2.book_id)
        test_lib.borrow_book(user.person_id, b3.book_id)
        test_lib.borrow_book(user.person_id, b4.book_id)
        print("Test 8 (Exceed Borrow Limit): FAIL")
    except ValueError:
        print("Test 8 (Exceed Borrow Limit Rule): PASS")

    try:
        user2 = test_lib.add_user("User 2")
        single_copy_book = test_lib.add_book("Rare Book", author.person_id, category.category_id, 1)
        test_lib.borrow_book(user2.person_id, single_copy_book.book_id)

        user3 = test_lib.add_user("User 3")
        test_lib.borrow_book(user3.person_id, single_copy_book.book_id)
        print("Test 9 (Zero Available Copies): FAIL")
    except ValueError:
        print("Test 9 (Zero Available Copies Edge Case): PASS")

    try:
        test_lib.delete_book(single_copy_book.book_id)
        print("Test 10 (Delete Borrowed Book): FAIL")
    except ValueError:
        print("Test 10 (Delete Borrowed Book Edge Case): PASS")

    print("========== TEST SUITE COMPLETED ==========\n")
run_test_scenarios()


========== STARTING AUTOMATED TEST SUITE ==========
Test 1 (Add Book): PASS
Test 2 (Add User): PASS
Test 3 (Borrow Book): PASS
Test 4 (Empty Title Validation): PASS
Test 5 (Invalid Copies Validation): PASS
Test 6 (Invalid User Validation): PASS
Test 7 (Borrow Same Book Twice Rule): PASS
Test 8 (Exceed Borrow Limit Rule): PASS
Test 9 (Zero Available Copies Edge Case): PASS
Test 10 (Delete Borrowed Book Edge Case): PASS
========== TEST SUITE COMPLETED ==========



In [5]:

import gradio as gr

library = Library()

if not library.authors:
    library.add_author("General Author")

if not library.categories:
    library.add_category("General Category")


def show_books(search_text="", status_filter="All"):
    library.load_data()
    books = library.books

    if search_text.strip():
        books = library.search_books(search_text)

    if status_filter != "All":
        books = [
            b for b in books
            if (status_filter == "Available" and b.available_copies > 0) or
               (status_filter == "Unavailable" and b.available_copies == 0)
        ]

    rows = []
    for book in books:
        try:
            author_name = library.get_author(book.author_id).name
        except:
            author_name = "Unknown"

        try:
            cat_name = library.get_category(book.category_id).name
        except:
            cat_name = "Unknown"

        rows.append([
            book.book_id,
            book.title,
            author_name,
            cat_name,
            book.copies,
            book.available_copies
        ])

    return rows


def add_user_gui(name):
    try:
        u = library.add_user(name)
        return f" User added successfully with ID: {u.person_id}"
    except Exception as error:
        return f" Error: {str(error)}"


def add_book_gui(title, author_id, category_id, copies):
    try:
        b = library.add_book(title, int(author_id), int(category_id), int(copies))
        return f" Book added successfully with ID: {b.book_id}"
    except Exception as error:
        return f" Error: {str(error)}"


def borrow_gui(user_id, book_id):
    try:
        br = library.borrow_book(int(user_id), int(book_id))
        return f"✅ Borrowed successfully! Borrowing ID: {br.borrowing_id}"
    except Exception as error:
        return f"❌ Error: {str(error)}"


def return_gui(borrowing_id):
    try:
        library.return_book(int(borrowing_id))
        return "✅ Book returned successfully!"
    except Exception as error:
        return f"❌ Error: {str(error)}"


def report_gui():
    data = library.report()
    return (
        f"📚 Total Books: {data['Total Books']}\n"
        f"📖 Available Copies: {data['Available Copies']}\n"
        f"👥 Users: {data['Users']}\n"
        f"🔄 Active Borrowings: {data['Active Borrowings']}"
    )


with gr.Blocks(title="Library System") as app:
    gr.Markdown("# 📖 Library Management System")

    with gr.Tab("📚 Books & Search"):
        with gr.Row():
            search_input = gr.Textbox(label="Search by Title or Author")
            status_filter = gr.Dropdown(["All", "Available", "Unavailable"], value="All", label="Filter Availability")

        search_btn = gr.Button("Search / Refresh Table")
        books_table = gr.Dataframe(
            headers=["ID", "Title", "Author", "Category", "Total Copies", "Available Copies"],
            datatype=["number", "str", "str", "str", "number", "number"]
        )
        search_btn.click(show_books, inputs=[search_input, status_filter], outputs=books_table)

    with gr.Tab("➕ Add User"):
        user_name = gr.Textbox(label="User Name")
        add_user_btn = gr.Button("Register User")
        user_msg = gr.Textbox(label="Status Response", interactive=False)
        add_user_btn.click(add_user_gui, inputs=user_name, outputs=user_msg)

    with gr.Tab("📘 Add Book"):
        book_title = gr.Textbox(label="Book Title")
        author_id = gr.Number(label="Author ID", value=1, precision=0)
        category_id = gr.Number(label="Category ID", value=1, precision=0)
        copies = gr.Number(label="Number of Copies", value=1, precision=0)
        add_book_btn = gr.Button("Save Book")
        book_msg = gr.Textbox(label="Status Response", interactive=False)
        add_book_btn.click(add_book_gui, inputs=[book_title, author_id, category_id, copies], outputs=book_msg)

    with gr.Tab("🔄 Borrow & Return"):
        gr.Markdown("### Borrow a Book")
        borrow_u_id = gr.Number(label="User ID", precision=0)
        borrow_b_id = gr.Number(label="Book ID", precision=0)
        borrow_btn = gr.Button("Confirm Borrow")
        borrow_msg = gr.Textbox(label="Borrow Status", interactive=False)
        borrow_btn.click(borrow_gui, inputs=[borrow_u_id, borrow_b_id], outputs=borrow_msg)

        gr.Markdown("---")
        gr.Markdown("### Return a Book")
        ret_id = gr.Number(label="Borrowing ID", precision=0)
        return_btn = gr.Button("Confirm Return")
        return_msg = gr.Textbox(label="Return Status", interactive=False)
        return_btn.click(return_gui, inputs=ret_id, outputs=return_msg)

    with gr.Tab("📊 Dashboard & Report"):
        report_btn = gr.Button("Fetch Current Summary Report")
        report_output = gr.Textbox(label="System Summary Report", interactive=False)
        report_btn.click(report_gui, outputs=report_output)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d25ce9ddc95c69abd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
